# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Reading from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

In [0]:
# goals:
# trim the string
# normalization for martial_status, gndr
# names are not friendly

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df =(
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status"))== "S", "Single")
         .when(F.upper(F.col("cst_marital_status"))== "M", "Married")
         .otherwise("n/a")
    )

    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr"))== "M", "Male")
         .when(F.upper(F.col("cst_gndr"))== "F", "Female")
         .otherwise("n/a")
    )
)


## Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

## Renaming columns

In [0]:
%python
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status":"marital_status",
    "cst_gndr":"gender",
    "cst_create_date": "created_date"
}

for oldname, newname in RENAME_MAP.items():
    df = df.withColumnRenamed(oldname, newname)


In [0]:
df.display()

# Write Into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.silver.crm_customers")
)

In [0]:
%sql

select * from workspace.silver.crm_customers